###### 25_support_ticket_agentic_ai
 
The purpose of this notebook is to build a Support Ticket Agentic AI application that intelligently selects and executes multiple AI tools (SQL Analytics, ML Prediction, Vector Search, and LLM reasoning) to answer support-related questions and generate grounded business recommendations.

###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- Databricks Foundation Model (databricks-meta-llama-3-1-8b-instruct)


###### Input

- User Question

- Delta table containing support ticket embeddings

- Existing Vector Search Endpoint

- Existing Vector Search Index

- Embedding Foundation Model

- Support Ticket Classifier Model Serving Endpoint

- LLM Foundation Model

- Tool-specific prompts


######  Output

- Grounded response generated by the selected AI tool

######  Architecture

```text

User Question
      │
      ▼
Tool Selection (LLM)
      │
      ▼
 Tool Registry
      │
 ┌────┼───────────┬───────────┐
 │    │           │           │
SQL  Similar   Prediction  Retention
Tool Tickets      Tool        Tool
 │    │           │           │
 └────┼───────────┼───────────┘
      │
      ▼
 Tool Result
      │
      ▼
 Tool-specific Prompt
      │
      ▼
 Databricks LLM
      │
      ▼
 Grounded Final Response
     
```

###### AI Workflow

1. User submits a question.
2. The LLM selects the most appropriate tool.
3. The selected tool executes its task:
    - SQL Analytics
    - Vector Search
    - ML Prediction
    - Retention Recommendation
4. A tool-specific prompt is created.
5. The LLM generates a grounded response using only the tool output.


###### Section 0 - Install libraries

In [0]:
# Install once if needed

%pip install databricks-vectorsearch
dbutils.library.restartPython()



###### Section 1 - Import Libraries

In [0]:
import requests

from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

###### Section 2 - Configuration

In [0]:

#source delta table
SOURCE_TABLE = "dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings"

#Vector Search endpoint name
VECTOR_SEARCH_ENDPOINT_NAME = "support-ticket-vector-search-endpoint"

#Vector index name
INDEX_NAME = (
    "dbw_agentic_ai_dev.support_ticket_ai."
    "gold_support_ticket_embeddings_index"
)

#Embedding model name
EMBEDDING_MODEL_NAME = "databricks-gte-large-en"

#LLM endpoint name
LLM_MODEL_NAME = "databricks-meta-llama-3-1-8b-instruct"

#Serving endpoint name
CLASSIFIER_ENDPOINT_NAME = "support-ticket-endpoint"



###### Section 3 - Clients

In [0]:

w = WorkspaceClient()

vector_search_client  = VectorSearchClient()

# Connect to Vector Search
index = vector_search_client.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)

###### Section 4 - LLM Helper

In [0]:
def generate_answer(prompt, max_tokens=300, temperature=0.0):
    response = w.serving_endpoints.query(
        name=LLM_MODEL_NAME,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )

    return response.choices[0].message.content

###### Section 5 - SQL Analytics Tool

In [0]:
#LLM selects from predefined SQL actions and then Python maps the selected action to approved SQL.

def select_sql_action(question):
    prompt = f"""
You are a SQL routing agent.

Choose the best SQL action for the user question.

Available actions:
1. count_tickets_by_category
2. total_support_tickets
3. distinct_categories
4. billing_tickets
5. login_tickets
6. technical_tickets
7. cancellation_tickets

Question:
{question}

Return only one action name.
"""

    return generate_answer(prompt, max_tokens=20).strip().lower()


Question
↓
LLM chooses action
↓
Python maps action → SQL

In [0]:
def execute_sql_action(action):
    sql_map = {
        "count_tickets_by_category": f"""
            SELECT category, COUNT(*) AS ticket_count
            FROM {SOURCE_TABLE}
            GROUP BY category
            ORDER BY ticket_count DESC
        """,

        "total_support_tickets": f"""
            SELECT COUNT(*) AS total_tickets
            FROM {SOURCE_TABLE}
        """,

        "distinct_categories": f"""
            SELECT DISTINCT category
            FROM {SOURCE_TABLE}
            ORDER BY category
        """,

        "billing_tickets": f"""
            SELECT ticket_id, cleaned_ticket_text, category
            FROM {SOURCE_TABLE}
            WHERE category = 'Billing'
        """,

        "login_tickets": f"""
            SELECT ticket_id, cleaned_ticket_text, category
            FROM {SOURCE_TABLE}
            WHERE category = 'Login'
        """,

        "technical_tickets": f"""
            SELECT ticket_id, cleaned_ticket_text, category
            FROM {SOURCE_TABLE}
            WHERE category = 'Technical'
        """,

        "cancellation_tickets": f"""
            SELECT ticket_id, cleaned_ticket_text, category
            FROM {SOURCE_TABLE}
            WHERE category = 'Cancellation'
        """
    }

    if action not in sql_map:
        return f"Unsupported SQL action: {action}"

    return spark.sql(sql_map[action]).collect()

In [0]:

def sql_analytics_tool(question):
    # Step 1 - Select the SQL action
    action = select_sql_action(question)

    # Step 2 - Execute the SQL query
    result = execute_sql_action(action)

    # Step 3 - Return the results
    return {
        "tool": "sql_analytics_tool",
        "action": action,
        "result": result
    }

###### Section 6 - Similar Support Tickets Tool

Question
↓
Embedding
↓
Vector Search
↓
Context
↓
Return

In [0]:

def similar_support_tickets_tool(question, num_results=3):
   
    # Step 1 - Convert the question into an embedding
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL_NAME,
        input=[question]
    )

    question_embedding = [
        float(x) for x in response.data[0].embedding
    ]

    # Step 2 - Query the Vector Search index
    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["ticket_id", "cleaned_ticket_text", "category"],
        num_results=num_results
    )

    similar_tickets = results["result"]["data_array"]
    
    # Step 3 - Build the retrieved context
    similar_ticket_context = []

    for ticket_id, cleaned_ticket_text, category, score in similar_tickets:
        similar_ticket_context.append(
            f"Support Ticket {ticket_id}: {cleaned_ticket_text} | Category: {category}"
        )

    retrieved_context = "\n".join(similar_ticket_context)

    # Step 4 - Return similar support tickets
    return {
        "tool": "similar_support_tickets_tool",
        "context": retrieved_context,
        "rows": similar_tickets
    }

###### Section 7 - Prediction Tool

Notebook
↓
REST call
↓
Serving Endpoint
↓
Prediction

In [0]:
def prediction_tool(question):

    # Step 1 - Build the serving endpoint URL
    workspace_url = (
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .apiUrl()
        .get()
    )

    endpoint_url = (
        f"{workspace_url}/serving-endpoints/"
        f"{CLASSIFIER_ENDPOINT_NAME}/invocations"
    )

    # Step 2 - Retrieve the authentication token
    token = dbutils.secrets.get(
        scope="support-ticket-agent-secrets",
        key="model-serving-pat"
    ).strip()

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Step 3 - Create the request payload
    payload = {
        "dataframe_records": [
            {
                "cleaned_ticket_text": question
            }
        ]
    }
    
    # Step 4 - Call the ML model serving endpoint
    response = requests.post(
        endpoint_url,
        headers=headers,
        json=payload
    )

    prediction = response.json()["predictions"][0]
 
    # Step 5 - Return the prediction
    return {
        "tool": "prediction_tool",
        "ticket_text": question,
        "prediction_response": prediction
    }

###### Section 8 - Retention Recommendation Tool

Prediction
↓
Vector Search
↓
LLM chooses action
↓
Python converts action
↓
Final recommendation

In [0]:
#create predefined recommendation actions, then let the LLM choose from them.
#LLM chooses recommendation action
#Python maps action → approved recommendation text
#LLM summarizes final answer


RETENTION_ACTIONS = {
    "offer_discount": (
        "Review the customer's account and offer an appropriate loyalty "
        "or retention discount based on eligibility."
    ),
    "offer_support_package": (
        "Contact the customer to understand the reason for cancellation, "
        "address any unresolved service issues, and offer complimentary "
        "priority support when appropriate."
    ),
    "service_quality_review": (
        "Escalate the case to the service quality team for an urgent review "
        "of connectivity, speed, reliability, or repeated technical issues."
    ),
    "billing_review": (
        "Assign a billing specialist to review disputed charges, explain "
        "the bill clearly, and correct any confirmed billing errors."
    ),
    "no_action": (
        "No immediate retention action is required. Continue normal support "
        "and monitor for additional dissatisfaction or cancellation signals."
    ),
}


In [0]:
#Choose the best retention action
def choose_retention_action(question, prediction, support_ticket_context):   
    
    prompt = f"""
You are a customer retention agent.

Choose the best retention action from the list.

Available actions:
1. offer_discount
2. offer_support_package
3. service_quality_review
4. billing_review
5. no_action

Support ticket:
{question}

Prediction:
{prediction}

Similar support ticket context:
{support_ticket_context}

Return only one action name.
"""

    return generate_answer(prompt, max_tokens=20).strip().lower()


In [0]:

def retention_recommendation_tool(question):

    # Step 1 - Predict the support ticket category
    prediction = prediction_tool(question)

    # Step 2 - Retrieve similar historical support tickets
    similar_result = similar_support_tickets_tool(question, num_results=3)    

    # Step 3 - Let the LLM choose the best retention action
    action = choose_retention_action(
        question = question,
        prediction = prediction,
        support_ticket_context = similar_result["context"]
    )
    
    # Step 4 - Convert the selected action into a recommendation
    recommendation = RETENTION_ACTIONS.get(
        action,
        RETENTION_ACTIONS["no_action"],
    )

    # Step 5 - Return the recommendation
    return {
        "tool": "retention_recommendation_tool",
        "question": question,
        "action": action,
        "recommended_action": recommendation,
        "prediction": prediction,
        "similar_context": similar_result["context"]
    }

###### Section 9 - Tool Selection

In [0]:
def choose_tool(question):
    prompt = f"""
You are an AI Support Ticket Agent.

Your job is to choose the single best tool to answer the user's question.

Available Tools

1. sql_analytics_tool
Purpose:
Answer analytical questions about the support ticket dataset.

Examples:
- How many Billing tickets?
- Show all Login tickets.
- What categories are available?

2. similar_support_tickets_tool
Purpose:
Find historical support tickets that are semantically similar to a user's issue.

Examples:
- Internet is slow.
- Refund not received.
- Login page keeps failing.

3. prediction_tool
Purpose:
Predict the support ticket category from new support ticket text.

Examples:
- Predict the category for:
  "Unable to log in."

- Classify:
  "Video streaming keeps buffering."

4. retention_recommendation_tool
Purpose:
Recommend the best customer support or retention action based on the support ticket text,
predicted category, and similar historical tickets.

Examples:
- What should we do for:
  "Please disconnect my service."

- Recommend the next action for:
  "Unexpected charges on my bill."

Question:
{question}

Return only one tool name.
"""

    return generate_answer(prompt, max_tokens=20).strip().lower()

###### Section 10 - Agent Function

In [0]:
#Tool Registry
TOOLS = {
    "sql_analytics_tool": sql_analytics_tool,
    "similar_support_tickets_tool": similar_support_tickets_tool,
    "prediction_tool": prediction_tool,
    "retention_recommendation_tool": retention_recommendation_tool
}

In [0]:
def build_final_prompt(question, selected_tool, tool_output):

    if selected_tool == "prediction_tool":
       return f"""
You are formatting the output of a machine learning classification tool.

Use ONLY the Tool Result.
Treat it as the single source of truth.

Return exactly in this format:

The model classified this support ticket as: <predicted_category>

Do not add introductions, explanations, or extra sentences.

User Question:
{question}

Tool Result:
{tool_output}
"""

    elif selected_tool == "sql_analytics_tool":
        return f"""
You are formatting the output of a SQL analytics tool.

Use ONLY the Tool Result.
Treat it as the single source of truth.

Return a concise business response.

Example:

Based on the SQL analytics results, there are <count> Billing tickets.

Do not add summaries or additional analysis.

User Question:
{question}

Tool Result:
{tool_output}
"""

    elif selected_tool == "similar_support_tickets_tool":
        return f"""
You are formatting semantic search results.

Use ONLY the Tool Result.
Treat it as the single source of truth.

Return exactly in this format:

Based on your question "<question>", the following support tickets are semantically similar:

- Ticket ...
- Ticket ...
- Ticket ...

Do not explain why they are similar.
Do not infer causes or solutions.
Do not add introductory or concluding sentences.

Tool Result:
{tool_output}
"""

    elif selected_tool == "retention_recommendation_tool":
       return f"""
You are formatting the output of a customer retention recommendation tool.

Use ONLY the Tool Result.
Treat it as the single source of truth.

For Recommended Action, use the value of `recommended_action`.
Do not display `recommended_action_key` or internal action names such as
`offer_support_package`, `billing_review`, or `no_action`.

Return exactly in this format:

**Recommended Action:**
<recommended_action>

**Predicted Category:**
<prediction>

**Similar Tickets:**
- <ticket1>
- <ticket2>
- <ticket3>

Rules:
- Do not add introductory or concluding sentences.
- Do not explain or justify the recommendation.
- Do not infer additional customer information.
- Preserve the order of similar tickets.
- Display only tickets returned by the tool.
- Output only the formatted answer.

User Question:
{question}

Tool Result:
{tool_output}
"""

In [0]:
def support_ticket_agent(question):
    # Step 1 - Choose the appropriate tool
    selected_tool = choose_tool(question)

    # Step 2 - Validate selected tool
    if selected_tool not in TOOLS:
        return {
            "question": question,
            "selected_tool": selected_tool,
            "error": "No valid tool selected."
        }

    # Step 3 - Execute selected tool
    tool_output = TOOLS[selected_tool](question)

    # Step 4 - Build tool-specific final prompt
    final_prompt = build_final_prompt(
        question=question,
        selected_tool=selected_tool,
        tool_output=tool_output
    )

    # Step 5 - Generate final answer
    final_answer = generate_answer(
        final_prompt,
        max_tokens=400
    )

    # Step 6 - Return response
    return {
        "question": question,
        "selected_tool": selected_tool,
        "tool_result": tool_output,
        "final_answer": final_answer
    }

###### Section 11 - Test Agent

In [0]:
test_questions = [
    
    "How many Billing tickets?",

    "Show tickets similar to internet is slow",

    "Predict the category for: Video streaming keeps buffering",

    "What should we do for: Please disconnect my service?"
]

for question in test_questions:
    result = support_ticket_agent(question)

    print("\nQUESTION:")
    print(result["question"])

    print("\nSELECTED TOOL:")
    print(result["selected_tool"])

    print("\nFINAL ANSWER:")
    print(result["final_answer"])

    print("-" * 80)

###### Notebook Summary

- Get Configurations for Vector Search endpoint name,  Vector index name,  Embedding model name and LLM endpoint name.

- Connect to Vector Search Index.

- Create Function to Call LLM.

- Implemented SQL Analytics Tool:
    - choose_sql_action
    - execute_sql_action

- Implemented Similar Ticket Retrieval Tool:
    - Convert the question into an embedding
    - Query the Vector Search index
    - Build the retrieved context
    - Return similar support tickets

- Implemented Prediction Tool:
    - Build the serving endpoint URL
    - Retrieve the authentication token
    - Create the request payload
    - Call the ML model serving endpoint
    - Return the prediction

- Implemented Retention Recommendation Tool:

    - Predict the support ticket category
    - Retrieve similar historical support tickets
    - Let the LLM choose the best retention action
    - Convert the selected action into a recommendation
    - Return the recommendation

- Implemented Agent Orchestration: 

    - support ticket agent
    - Choose the appropriate tool
    - Validate selected tool
    - Execute selected tool
    - Generate final answer
    - Return response

- Test Agent 

- What did we build?

    - A multi-tool Agentic AI application that intelligently selects between SQL analytics, semantic search, machine learning prediction, and retention recommendation tools to answer support ticket questions.



###### Key Learnings

- Built an ML model that classifies support tickets using a Databricks Model Serving endpoint.
- Used Databricks Vector Search to retrieve semantically similar historical support tickets.
- Built a grounded Retrieval-Augmented Generation (RAG) pipeline using Vector Search and an LLM.
- Developed a multi-tool Agentic AI application that dynamically selects the appropriate tool based on the user's question.
- Designed tool-specific prompts to generate grounded responses and reduce LLM hallucinations.
- Learned how to securely authenticate Model Serving endpoints using Databricks Secret Scopes.
- Learned how to troubleshoot Model Serving authentication and API access issues in a production-style environment.
- Learned how to combine multiple AI capabilities (SQL, ML, Vector Search, and LLM reasoning) into a single Agentic AI workflow using intelligent tool selection.

###### Notebook Conclusion

- This notebook demonstrates how an Agentic AI system can intelligently orchestrate multiple specialized AI tools instead of relying on a single LLM. By combining SQL Analytics, Vector Search, Machine Learning prediction, and LLM reasoning, the agent selects the appropriate tool for each user request and produces grounded, context-aware responses. This design reduces hallucinations, improves response quality, and reflects the architecture commonly used in production AI systems.

- We also introduced tool-specific prompts so that each tool generates responses using only its own results, improving grounding and reducing hallucinations.

- This notebook demonstrates how multiple AI components can work together to build a production-style Agentic AI workflow.

###### Next Notebook